In [7]:
# Jupyter setup to expand cell display to 100% width on your screen (optional)
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

/var/folders/7w/k64x4hxx1x96s_hxn35hcg0w0000gp/T/ipykernel_92161/477910912.py:2: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


In [8]:
# Import relevant modules and setup for calling glmnet
%reset -f
%matplotlib inline

import sys
sys.path.append('../test')
sys.path.append('../lib')
import scipy, importlib, pprint, matplotlib.pyplot as plt, warnings
from glmnet import glmnet; from glmnetPlot import glmnetPlot
from glmnetPrint import glmnetPrint; from glmnetCoef import glmnetCoef; from glmnetPredict import glmnetPredict
from cvglmnet import cvglmnet; from cvglmnetCoef import cvglmnetCoef
from cvglmnetPlot import cvglmnetPlot; from cvglmnetPredict import cvglmnetPredict

In [24]:
total_channel_names = ['GeoNO2_24Hours', 'GCHP_NO2_24Hours', 'GCHP_PMIDDRY_24Hours',
                        'NO_CEDS_anthro_emi', 'NMVOC_CEDS_anthro_emi', 'NH3_CEDS_anthro_emi',
                        'Total_DM',
                        'NDVI', 'ISA',
                        'major_roads', 'minor_roads',
                        'forests_density', 'shrublands_density', 'croplands_density', 'urban_builtup_lands_density', 'water_bodies_density',
                        'V10M', 'U10M', 'T2M', 'RH', 'PBLH', 'PRECTOT',
                        'Lat', 'Lon', 'elevation',
                        'Population']

Training_dir = '/Volumes/rvmartin2/Active/yany1/1.project/NO2_DL_US_2019/TrainingDatasets/US_NO2-v0.0.0/'
training_infile = 'cnn_TrainingData_189channels_11x11_201901-201912.nc'
geophysical_biases_data_dir = '/Volumes/rvmartin2/Active/yany1/1.project/NO2_DL_US_2019/data/monthly_bias/'
geophysical_biases_data_infile = 'NO2_24Hours_monthly_bias_v0.0.0.nc'

In [18]:
import xarray as xr
# load data
x = xr.open_dataset(Training_dir + training_infile)
y = xr.open_dataset(geophysical_biases_data_dir + geophysical_biases_data_infile)

In [ ]:
# create weights
t = scipy.ones((50, 1), dtype = scipy.float64)
wts = scipy.row_stack((t, 2*t))
# call glmnet
fit = glmnet(x = x.copy(), y = y.copy(), family = 'gaussian', \
                    weights = wts, \
                    alpha = 0.2, nlambda = 20
                    )

In [12]:
# Import necessary libraries
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import matplotlib.pyplot as plt
import os

# Define a function to calculate VIF
def calculate_vif(df):
    """
    Calculate VIF for each feature in a dataframe
    
    Args:
        df (pandas.DataFrame): DataFrame containing feature variables
        
    Returns:
        pandas.DataFrame: DataFrame with feature names and VIF values
    """
    # Create a dataframe with constant
    X = add_constant(df)
    
    # Calculate VIF for each feature
    vif_data = pd.DataFrame()
    vif_data['Feature'] = X.columns
    vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    
    # Sort by highest VIF
    vif_data = vif_data.sort_values('VIF', ascending=False)
    
    return vif_data

In [25]:
available_channels = []
for channel in total_channel_names:
    if channel in x:
        available_channels.append(channel)
    else:
        print(f"Channel {channel} not found in x")

# Create a pandas DataFrame for VIF calculation
# We'll extract either the center pixel or the mean of each grid
# Option 1: Center pixel (for 11x11 grid, center is at [5,5])
center_x, center_y = 1, 1  # Center coordinates

# Create empty DataFrame
df_features = pd.DataFrame()

# Extract center pixel value for each feature and each observation
for channel in available_channels:
    # Get the center pixel value for each observation
    center_values = x[channel].isel(width=center_x, height=center_y).values
    df_features[channel] = center_values

# Alternative approach: calculate mean of each grid
df_features_mean = pd.DataFrame()
for channel in available_channels:
    # Calculate mean across width and height dimensions for each observation
    mean_values = x[channel].mean(dim=['width', 'height']).values
    df_features_mean[f"{channel}_mean"] = mean_values

# Remove rows with NaN values
df_features = df_features.dropna()
df_features_mean = df_features_mean.dropna()

# Calculate VIF for center pixel approach
print("VIF calculation using center pixel values:")
vif_results = calculate_vif(df_features)
print(vif_results)

VIF calculation using center pixel values:
                        Feature          VIF
0                         const  4957.201039
5         NMVOC_CEDS_anthro_emi     7.330065
2              GCHP_NO2_24Hours     5.245312
20                           RH     4.359514
4            NO_CEDS_anthro_emi     4.341980
1                GeoNO2_24Hours     4.300307
24                          Lon     3.861186
15  urban_builtup_lands_density     3.517544
26                   Population     3.193802
9                           ISA     3.183972
19                          T2M     2.795064
21                         PBLH     2.652746
3          GCHP_PMIDDRY_24Hours     2.537573
25                    elevation     2.378515
23                          Lat     2.065809
16         water_bodies_density     2.051860
11                  minor_roads     1.995288
22                      PRECTOT     1.777555
6           NH3_CEDS_anthro_emi     1.759241
17                         V10M     1.702192
10          

In [ ]:
glmnetPrint(fit)

Perform Lasso variable selection workflow from NetCDF file.

Args:
    TrainingDatasets (xarray.Dataset): Training datasets.
    target_name (str): Name of the target variable.
    expected_sign (dict or None): Dict mapping feature names to sign constraints: 1 (positive), -1 (negative), 0 (no constraint).
    n_folds (int): Number of folds for cross-validation.
    random_state (int): Random seed.
    output_summary (bool): Whether to print summary statistics.

Returns:
    dict: Results including selected variables, coefficients, OLS summary, VIF, and adjusted R^2.